# CREMA-D Mirror: Clone + Git LFS Download (Step-by-step)

This notebook is paired with `nav.md`.

How to use this notebook:
- Run cells from top to bottom.
- If a cell errors, stop and follow the error message.

Outcome:
- You end up with real `*.mp3`, `*.wav`, `*.flv` files on disk (not Git LFS pointer stubs).


## 0) Python dependencies (pip)

This notebook only needs `pandas` for a couple sanity checks.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install pandas

## 1) Choose where to clone the repo

Set `WORKDIR` to a folder where you have a few GB free.


In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/cs-cooper-lab/crema-d-mirror.git"

# Change this if you want to clone somewhere else.
WORKDIR = Path.cwd().resolve()

# This is where the repo will live on disk.
CLONE_DIR = WORKDIR / "crema-d-mirror"

WORKDIR, CLONE_DIR

## 2) Clone the mirror repo (idempotent)

This will clone only if `CLONE_DIR` is not already a git repo.


In [ ]:
if not (CLONE_DIR / ".git").exists():
    !git clone "{REPO_URL}" "{CLONE_DIR}"
else:
    print("Repo already cloned:", CLONE_DIR)

## 3) Basic repo sanity check


In [ ]:
REPO_DIR = CLONE_DIR

expected = [
    REPO_DIR / "AudioMP3",
    REPO_DIR / "AudioWAV",
    REPO_DIR / "VideoFlash",
    REPO_DIR / "SentenceFilenames.csv",
]

missing = [p for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError("Missing expected paths:\n" + "\n".join(str(p) for p in missing))

print("Repo root:", REPO_DIR)
print("OK: expected folders/files exist")

## 4) Confirm whether media files are Git LFS pointers right now

If you see `version https://git-lfs.github.com/spec/v1`, the file is a pointer stub (not real audio).


In [ ]:
def read_head(path: Path, nbytes: int = 256) -> str:
    return path.read_text("utf-8", errors="replace")[:nbytes]

def is_lfs_pointer(path: Path) -> bool:
    return "git-lfs.github.com/spec" in read_head(path, 256)

sample_mp3 = REPO_DIR / "AudioMP3" / "1001_DFA_ANG_XX.mp3"
print("Sample file:", sample_mp3)
print(read_head(sample_mp3, 200))
print("Is LFS pointer?", is_lfs_pointer(sample_mp3))

## 5) Preflight: Git LFS must be installed

This cell checks whether Git LFS is available.

If it errors, install Git LFS (one-time per machine) and then re-run this cell:
- macOS (Homebrew): `brew install git-lfs`
- Debian/Ubuntu: `sudo apt-get update && sudo apt-get install git-lfs`
- Windows: https://git-lfs.com/ (or `choco install git-lfs`)


In [ ]:
import subprocess

res = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(
        "Git LFS is not installed or not on PATH.\n\n"
        "Install it, then restart your terminal/Jupyter and re-run this cell.\n\n"
        "macOS (Homebrew): brew install git-lfs\n"
        "Debian/Ubuntu: sudo apt-get update && sudo apt-get install git-lfs\n"
        "Windows: https://git-lfs.com/ (or choco install git-lfs)\n\n"
        f"git lfs version output:\n{res.stdout}{res.stderr}"
    )

print(res.stdout.strip())

## 6) Enable Git LFS for your user (safe to run multiple times)


In [ ]:
!git -C "{REPO_DIR}" lfs install

## 7) Download the real media files into your clone

This step can take a while and downloads multiple GB.


In [ ]:
!git -C "{REPO_DIR}" lfs pull

## 8) Verify media is real (not pointers) + quick mapping sanity check


In [ ]:
print("Is LFS pointer now?", is_lfs_pointer(sample_mp3))
print("On-disk size (bytes):", sample_mp3.stat().st_size)
print("First 80 chars:\n", read_head(sample_mp3, 80))

if is_lfs_pointer(sample_mp3):
    raise RuntimeError(
        "Files are still Git LFS pointers after `git lfs pull`.\n"
        "Go to the Troubleshooting section at the bottom of this notebook and run those commands."
    )

In [ ]:
import pandas as pd

# Pick one real stimulus from the master index and check the three expected files exist.
sf = pd.read_csv(REPO_DIR / "SentenceFilenames.csv")
row = sf.sample(1, random_state=0).iloc[0]
stem = row["Filename"]

paths = {
    "mp3": REPO_DIR / "AudioMP3" / f"{stem}.mp3",
    "wav": REPO_DIR / "AudioWAV" / f"{stem}.wav",
    "flv": REPO_DIR / "VideoFlash" / f"{stem}.flv",
}

print("Random stimulus:")
print("  Stimulus_Number:", int(row["Stimulus_Number"]))
print("  Filename:", stem)
print("Paths:")
for k, p in paths.items():
    print(f"  {k}: {p} (exists={p.exists()}, size={p.stat().st_size if p.exists() else 'n/a'})")

## Troubleshooting (only if step 8 failed)

If media files are still pointers after `git lfs pull`, run these commands and then re-run steps 7 and 8:

```bash
git -C crema-d-mirror lfs fetch --all
git -C crema-d-mirror lfs checkout
```

If you only want a subset (saves time/disk):

```bash
# Audio only
git -C crema-d-mirror lfs fetch --include="AudioMP3/*,AudioWAV/*"
git -C crema-d-mirror lfs checkout --include="AudioMP3/*,AudioWAV/*"

# Video only
git -C crema-d-mirror lfs fetch --include="VideoFlash/*"
git -C crema-d-mirror lfs checkout --include="VideoFlash/*"
```

If you downloaded a zip at any point:
- You cannot pull LFS objects from a zip; you must `git clone` with Git LFS installed.

After setup is complete, use `nav.md` to understand how CSV rows map to files.
